In [ ]:
visualization_target = input("INPUT 'visualization_target RUN_ID'(e.g., run_id_6): ")
model_in_run = input("INPUT 'model_in_run'(e.g., bert_based, lda, tag): ")

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

from setting_for_sda.path_setting import path_list
from setting_for_sda.date_setting import Date_Setting


import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from lib.visualization.distribution_collector import collect_top_bottom_tags
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()




In [ ]:
viz_dir = f'{path_list["data_root_dir"]}/result/{model_in_run}/{visualization_target}'
data_dir = f"{viz_dir}/data"
option_dict = load_json(f"{viz_dir}/option.json")


output_dir = './fig/'
date_range = 'Weekly'

std_date = datetime.strptime(Date_Setting.chatgpt_release_date, "%Y.%m.%d")
# pre_std_date = datetime.datetime(2021, 12, 1)


In [ ]:
df = load_df(data_dir, ['cdate' , 'id' , 'tag', 'cnt', 'tot_cnt', 'pct'])

In [ ]:
df['cdate'] = pd.to_datetime(df['cdate'], format="%Y-%m-%d")
df['rel_week'] = np.floor((df['cdate']-std_date).dt.days/7)

In [ ]:
proportion_dict = dict()
proportion_dict = collect_top_bottom_tags(df)

In [ ]:
proportion_dict

In [ ]:
color_list = ["#4C704C", "#A3C9A8"]

In [ ]:
proportion_dict.keys()

In [ ]:
sharey = False ## 또는 sharey=False
sharex = True ## 또는 sharex=False

fig, axs = plt.subplots(1, 2, figsize = (18, 6), constrained_layout=True)
alpha_list = [0.6, 0.5]
color_list = ["#a6d96a", "#1a9850"]

for idx, (title, proportion) in enumerate(proportion_dict.items()):
    n_len = len(proportion_dict.items())
    rel_week = list(proportion['rel_week'])
    values = list(proportion['pct_pct'])


    axs[idx].bar(rel_week, values, color=color_list[idx], width=1.0, align='center', alpha=alpha_list[idx]
    )
    axs[idx].axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)

    if idx ==2 :
        axs[idx].set_ylim(0.85, 1.0)
        axs[idx].set_yticks(np.arange(0.85, 1.01, 0.05))

    axs[idx].set_title(f'{title} for {option_dict["selected_tags"]}', fontsize=25)
    axs[idx].tick_params(axis='x', labelsize=16)
    axs[idx].tick_params(axis='y', labelsize=16)



axs[0].set_ylabel("Accumulated tag share", fontsize = 22)

fig.supxlabel("Week relative to ChatGPT release", fontsize=22) 
plt.savefig(f"{output_dir}C_Result_Fig2_1.png", dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
df

In [ ]:
# std_date = datetime.datetime(2022, 11, 30)
# tag_distribution_dict = dict()


tot_df = df.groupby(['rel_week'])['pct'].sum().reset_index(name='tot_pct')
tag_dis_by_df = df.groupby(['rel_week', 'tag'])['cnt'].sum().reset_index(name = 'pct')

df_proportion = pd.merge(tag_dis_by_df, tot_df, on = ['rel_week'], how = 'left')
df_proportion['proportion'] = df_proportion['pct'] / df_proportion['tot_pct']




In [ ]:
df_proportion

In [ ]:
entropy = (
    df_proportion
    .groupby('rel_week')['proportion']
    .apply(lambda x: calculate_entropy(x.values))
    .reset_index()
    .sort_values('rel_week')
    .proportion
)

gini = (
    df_proportion
    .groupby('rel_week')['proportion']
    .apply(lambda x: calculate_gini(x.values))
    .reset_index()
    .sort_values('rel_week')
    .proportion
)

In [ ]:
rel_week = np.array(np.arange(-52, 156))

In [ ]:
proportion_dict

In [ ]:
def plot_regression(ax, values, title, ylabel=None):
    
    list_ = list(values)
    x_rel, divider = get_dist_x_param_div(list_, 52)

    reg_bf = calc_regression_with_ci(x_rel[:divider], list_[:divider])
    reg_af = calc_regression_with_ci(x_rel[divider:], list_[divider:])

    bf = reg_bf["pred_summary"]
    af = reg_af["pred_summary"]

    ax.scatter(x_rel, list_, color='darkgray', alpha=0.7, s=10, marker='x')
    ax.axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)

    ax.plot(x_rel[:divider], bf["mean"], linewidth=2, label='before ChatGPT')
    ax.plot(x_rel[divider:], af["mean"], linewidth=2, label='after ChatGPT')

    ax.fill_between(x_rel[:divider], bf["mean_ci_lower"], bf["mean_ci_upper"], alpha=0.1)
    ax.fill_between(x_rel[divider:], af["mean_ci_lower"], af["mean_ci_upper"], alpha=0.1)

    ax.legend(frameon=False, loc='best', fontsize=14)

    ax.text(
        0.5, 1.05, title,
        ha='center', va='bottom',
        fontsize=22, fontweight='bold',
        transform=ax.transAxes
    )

    if ylabel:
        ax.set_ylabel(ylabel, fontsize=22)

    ax.tick_params(axis='x', labelsize=16)
    ax.tick_params(axis='y', labelsize=16)

In [ ]:
sharey = False ## 또는 sharey=False
sharex = True ## 또는 sharex=False

fig, axs = plt.subplots(1, 4, figsize = (24, 6), constrained_layout=True)
alpha_list = [0.6, 0.5]
color_list = ["#a6d96a", "#1a9850"]

for x, (title, proportion) in enumerate(proportion_dict.items()):
    rel_week = list(proportion['rel_week'])
    values = list(proportion['pct_pct'])
    
    axs[x].bar(rel_week, values, color=color_list[x], width=1.0, align='center', alpha=alpha_list[x]
    )
    axs[x].axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)

    if x ==0 :
        axs[x].set_ylim(0.85, 1.0)
        axs[x].set_yticks(np.arange(0.85, 1.01, 0.05))


    axs[x].text(0.5, 1.05, f'{title} for {option_dict["selected_tags"]}',
            ha='center', va='bottom', fontsize=22, fontweight='bold', transform=axs[x].transAxes)

    axs[x].text(0.5, 1.00, "",
        ha='center', va='bottom', fontsize=15, transform=axs[x].transAxes)  
    axs[x].tick_params(axis='x', labelsize=16)
    axs[x].tick_params(axis='y', labelsize=16)


plot_regression(
    axs[2],
    entropy.values,
    f"Changes in Entropy (tag) for {option_dict['selected_tags']}",
    ylabel="Entropy"
)

plot_regression(
    axs[3],
    gini.values,
    f"Changes in Gini (tag) for {option_dict['selected_tags']}",
    ylabel="Gini"
)



axs[idx].tick_params(axis='x', labelsize=16)
axs[idx].tick_params(axis='y', labelsize=16)


axs[0].set_ylabel("Accumulated tag share", fontsize = 22)
axs[2].set_ylabel(f"Entropy", fontsize = 22)

fig.supxlabel("Weeks relative to ChatGPT release", fontsize=22) 
plt.savefig(f"{output_dir}C_Result_Fig2_2.png", dpi=300, bbox_inches='tight')
plt.show();